# FedTrace — 05: Grant Outlays via Bulk Download

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/05_grant_outlays_bulk.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~45 min (first run, includes download time)  
**Input checkpoints:** `data/raw/02_grants.jsonl` — written by notebook 02  
**Publishes to GitHub:** `data/raw/05_grant_outlays.jsonl`, `data/outputs/05_grant_outlays.json`, `data/findings/05_grant_outlays.md`

**Context:** All USASpending API endpoints return no outlay data for cancelled assistance awards (notebook 04). The bulk download files (`Assistance_PrimeTransactions`) are direct database exports, not filtered by search-index status — they include cancelled awards.

**Driving questions:**
1. Do the USASpending bulk download files include outlay columns for assistance awards?
2. After filtering to our 12,361 FAINs, what fraction have outlay data present?
3. What are the aggregate outlay totals for cancelled grants?

**Design:**
- Section 2 (Probe): download one small filtered file (NSF, 3 months) via the bulk download API to inspect the file schema and confirm outlay column names.
- Section 3 (Full Fetch): download pre-built annual archive files directly from `files.usaspending.gov` (static CDN — no API generation step, no dropped connections). Stream-parse FY2020–FY2026 in chunks, filter to our FAIN set, save matched rows. Intermediate files deleted after parsing.

Current findings: [`data/findings/05_grant_outlays.md`](../data/findings/05_grant_outlays.md).

## Setup

Run cells 1–4 at the start of every session. They are idempotent — safe to re-run.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('05_grant_outlays_bulk', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import io
import json
import time
import zipfile
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

USA_BASE     = 'https://api.usaspending.gov/api/v2'
DOWNLOAD_DIR = Path('/content/bulk_downloads')   # outside repo — not committed
DOWNLOAD_DIR.mkdir(exist_ok=True)

# Chunk size for streaming CSV parse (rows)
CHUNK_ROWS = 50_000

_RETRYABLE = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_jsonl_ids(path: Path, id_field: str) -> set:
    path = Path(path)
    if not path.exists():
        return set()
    ids = set()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                ids.add(json.loads(line).get(id_field))
    return ids


def append_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, 'a') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')


def extract_fain(generated_unique_award_id: str) -> str | None:
    """Extract FAIN from ASST_{TYPE}_{FAIN}_{AGENCY_CODE} format."""
    parts = str(generated_unique_award_id).split('_')
    if len(parts) < 4 or parts[0] != 'ASST':
        return None
    return '_'.join(parts[2:-1]) or None


def request_bulk_download(
    filters: dict,
    columns: list[str] | None = None,
    file_format: str = 'csv',
) -> dict:
    """Submit a bulk download request and return the status + file metadata."""
    payload: dict = {'filters': filters, 'file_format': file_format}
    if columns:
        payload['columns'] = columns
    for attempt in range(6):
        wait = 2 ** attempt
        try:
            r = requests.post(f'{USA_BASE}/bulk_download/awards/', json=payload, timeout=60)
            if r.status_code == 429:
                time.sleep(wait)
                continue
            if r.status_code != 200:
                raise RuntimeError(f'Bulk download request failed ({r.status_code}): {r.text[:400]}')
            return r.json()
        except _RETRYABLE as e:
            print(f'  Connection error (attempt {attempt+1}/6): {e}')
            time.sleep(wait)
    raise RuntimeError('Max retries exceeded on bulk download request')


def poll_bulk_download(status_url: str, poll_interval: int = 10, timeout_minutes: int = 30) -> dict:
    """Poll status_url until status is finished. Returns final status response."""
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        r = requests.get(status_url, timeout=30)
        data = r.json()
        status = data.get('status', 'unknown')
        print(f'  Status: {status}', end='\r')
        if status == 'finished':
            print(f'  Status: finished             ')
            return data
        if status in ('failed', 'error'):
            raise RuntimeError(f'Bulk download failed: {data}')
        time.sleep(poll_interval)
    raise TimeoutError(f'Bulk download did not finish within {timeout_minutes} minutes')


def download_and_extract(file_url: str, dest_dir: Path) -> list[Path]:
    """Download ZIP from file_url, extract to dest_dir, return extracted CSV paths."""
    zip_path = dest_dir / 'download.zip'
    print(f'Downloading {file_url} ...')
    with requests.get(file_url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f'  {downloaded/1e6:.1f} / {total/1e6:.1f} MB', end='\r')
    print(f'  Downloaded: {downloaded/1e6:.1f} MB            ')

    csv_paths = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith('.csv'):
                zf.extract(name, dest_dir)
                csv_paths.append(dest_dir / name)
    zip_path.unlink()  # free disk space
    print(f'  Extracted: {[p.name for p in csv_paths]}')
    return csv_paths


def filter_csv_to_fains(
    csv_path: Path,
    fain_set: set[str],
    fain_col: str,
    outlay_col: str | None,
    obligation_col: str | None,
) -> pd.DataFrame:
    """Stream-parse a CSV in chunks, returning rows whose FAIN is in fain_set."""
    matched_chunks = []
    total_rows = 0
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK_ROWS, dtype=str, low_memory=False):
        total_rows += len(chunk)
        if fain_col not in chunk.columns:
            print(f'  Warning: FAIN column "{fain_col}" not in CSV. Available: {list(chunk.columns[:10])}')
            return pd.DataFrame()
        mask = chunk[fain_col].isin(fain_set)
        if mask.any():
            matched_chunks.append(chunk[mask])
        print(f'  Parsed {total_rows:,} rows, {sum(len(c) for c in matched_chunks):,} matched', end='\r')
    print(f'  Parsed {total_rows:,} rows total, {sum(len(c) for c in matched_chunks):,} matched')
    return pd.concat(matched_chunks, ignore_index=True) if matched_chunks else pd.DataFrame()

ARCHIVE_BASE = 'https://files.usaspending.gov/award_data_archive'


def archive_file_urls(fiscal_year: int) -> list[str]:
    """Return URLs for all numbered PrimeTransactions files for a fiscal year.

    USASpending splits large years into numbered files (_1, _2, ...). Probe
    file numbers 1–9; stop at first 404. FY = Oct 1 (prev year) – Sep 30.
    """
    urls = []
    for n in range(1, 10):
        url = f'{ARCHIVE_BASE}/FY{fiscal_year}_All_Assistance_PrimeTransactions_{n}.csv.zip'
        try:
            head = requests.head(url, timeout=15, allow_redirects=True)
            if head.status_code == 200:
                size_mb = int(head.headers.get('Content-Length', 0)) / 1e6
                urls.append({'url': url, 'size_mb': round(size_mb, 1)})
            else:
                break  # no more numbered files for this year
        except _RETRYABLE:
            break
    return urls


def download_archive_file(url: str, dest_path: Path) -> Path:
    """Stream-download a ZIP to dest_path. Returns dest_path."""
    for attempt in range(4):
        try:
            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get('Content-Length', 0))
                downloaded = 0
                with open(dest_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=2 * 1024 * 1024):
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            print(f'  {downloaded/1e6:.0f}/{total/1e6:.0f} MB', end='\r')
            print(f'  Downloaded: {downloaded/1e6:.1f} MB          ')
            return dest_path
        except _RETRYABLE as e:
            print(f'  Connection error (attempt {attempt+1}/4): {e}')
            if dest_path.exists():
                dest_path.unlink()
            time.sleep(2 ** attempt)
    raise RuntimeError(f'Failed to download {url} after 4 attempts')


In [ ]:
# ── CELL 4: Load grants & build FAIN set ──────────────────────────────────────
grants_path        = PATHS['raw_dir'] / '02_grants.jsonl'
grant_outlays_path = PATHS['raw_dir'] / '05_grant_outlays.jsonl'
output_json_path   = PATHS['outputs_dir'] / '05_grant_outlays.json'
findings_md_path   = PATHS['data_root'] / 'findings' / '05_grant_outlays.md'

grants_records = load_jsonl(grants_path)
grants_df      = pd.DataFrame(grants_records)
grants_df['fain'] = grants_df['award_id'].apply(
    lambda x: extract_fain(str(x)) if pd.notna(x) else None
)

fain_set   = set(grants_df['fain'].dropna())
fain_to_id = dict(zip(grants_df['fain'].dropna(), grants_df.loc[grants_df['fain'].notna(), 'award_id']))

# Top agencies for download strategy
agency_counts = (
    grants_df.groupby(grants_df.get('awarding_agency', grants_df.get('agency', pd.Series())))
    .size().sort_values(ascending=False)
)

print(f'Grants loaded:          {len(grants_df):,}')
print(f'FAINs extracted:        {len(fain_set):,}')
print(f'Already fetched:        {len(load_jsonl_ids(grant_outlays_path, "award_id")):,}')
print()
print('Top agencies (for download planning):')
for ag, n in agency_counts.head(10).items():
    print(f'  {n:5d}  {ag}')

## 2. Probe: Schema Inspection via Filtered Bulk Download

Before downloading years of data, confirm two things:
1. The bulk download API accepts our filter parameters without errors
2. The resulting CSV contains an outlay column (and identify its exact name)

Probe filter: NSF grants, 3-month window (2024-01-01 to 2024-03-31). NSF has 1,474 of our grants, and this narrow date range keeps the probe file small (~MB, not GB).

If the probe returns no rows for our FAINs, the bulk download file likely uses a different FAIN key or normalizes the identifier — check what the `award_id_fain` field contains and adjust `fain_col` accordingly.

In [ ]:
# ── Probe filter — narrow window to keep file small ───────────────────────────
PROBE_AGENCY     = 'National Science Foundation'
PROBE_START_DATE = '2024-01-01'
PROBE_END_DATE   = '2024-03-31'

print(f'Requesting bulk download: {PROBE_AGENCY}, {PROBE_START_DATE} to {PROBE_END_DATE}')

probe_filters = {
    'prime_award_types': ['02', '03', '04', '05', '06', '09', '10', '11'],
    'agencies': [{'type': 'awarding', 'tier': 'toptier', 'name': PROBE_AGENCY}],
    'date_type': 'action_date',
    'date_range': {'start_date': PROBE_START_DATE, 'end_date': PROBE_END_DATE},
}

try:
    probe_meta = request_bulk_download(probe_filters)
    print(f'Request accepted.')
    print(f'Response keys: {list(probe_meta.keys())}')
    print(json.dumps(probe_meta, indent=2))
except RuntimeError as e:
    print(f'Request failed: {e}')
    probe_meta = None

In [ ]:
if not probe_meta:
    raise RuntimeError('Probe request failed — fix the filter parameters above before continuing.')

status_url = probe_meta.get('status_url') or probe_meta.get('url')
if not status_url:
    # Some responses include file_url directly if the file is pre-built
    file_url = probe_meta.get('file_url')
    if not file_url:
        raise RuntimeError(f'No status_url or file_url in response: {probe_meta}')
    print(f'File URL returned directly: {file_url}')
else:
    print(f'Polling: {status_url}')
    status_data = poll_bulk_download(status_url)
    file_url    = status_data.get('file_url') or status_data.get('url')

print(f'File URL: {file_url}')

In [ ]:
probe_dir  = DOWNLOAD_DIR / 'probe'
probe_dir.mkdir(exist_ok=True)

probe_csvs = download_and_extract(file_url, probe_dir)
print(f'\nCSV files: {[p.name for p in probe_csvs]}')

In [ ]:
# ── Inspect schema — find FAIN column and outlay columns ──────────────────────
# Focus on the assistance prime transactions file
assistance_csvs = [
    p for p in probe_csvs
    if 'assistance' in p.name.lower() or 'grant' in p.name.lower() or 'prime' in p.name.lower()
] or probe_csvs  # fallback: use all files

for csv_path in assistance_csvs:
    print(f'\n=== {csv_path.name} ===')
    # Read just the header + first 5 rows
    sample = pd.read_csv(csv_path, nrows=5, dtype=str)
    print(f'Columns ({len(sample.columns)}):')
    for col in sample.columns:
        sample_val = str(sample[col].iloc[0]) if not sample.empty else ''
        print(f'  {col:50s} {sample_val[:60]}')

    # Specifically look for FAIN and outlay columns
    fain_candidates   = [c for c in sample.columns if 'fain' in c.lower() or ('award' in c.lower() and 'id' in c.lower())]
    outlay_candidates = [c for c in sample.columns if 'outlay' in c.lower()]
    obligated_candidates = [c for c in sample.columns if 'obligat' in c.lower()]

    print(f'\n  FAIN candidates:       {fain_candidates}')
    print(f'  Outlay candidates:     {outlay_candidates}')
    print(f'  Obligation candidates: {obligated_candidates}')

In [ ]:
# ── Test FAIN matching against our set ────────────────────────────────────────
# Update these based on the schema inspection above.
# Defaults are the standard USASpending DATA Act field names.
FAIN_COL       = 'award_id_fain'          # adjust if schema inspection shows a different name
OUTLAY_COL     = 'federal_outlay'         # adjust if schema shows a different name
OBLIGATION_COL = 'federal_action_obligation'

# NSF FAINs from our grants for the probe window
nsf_fains = set(
    grants_df.loc[
        (grants_df.get('awarding_agency', grants_df.get('agency', '')) == PROBE_AGENCY)
        & grants_df['fain'].notna(),
        'fain'
    ]
)
print(f'NSF FAINs in our grant set: {len(nsf_fains)}')

for csv_path in assistance_csvs:
    print(f'\nMatching against {csv_path.name}:')
    matched = filter_csv_to_fains(csv_path, nsf_fains, FAIN_COL, OUTLAY_COL, OBLIGATION_COL)

    if matched.empty:
        print('  No FAIN matches found in probe file.')
        print('  Possible causes:')
        print('  1. Wrong FAIN_COL name — update above')
        print('  2. FAIN normalization difference (case, leading zeros)')
        print('  3. These grants were not active/modified in the probe date window')
        print(f'  Available columns: {list(pd.read_csv(csv_path, nrows=0).columns)}')
    else:
        print(f'  Matched {len(matched)} rows from {matched[FAIN_COL].nunique()} unique FAINs')

        if OUTLAY_COL in matched.columns:
            outlay_vals = pd.to_numeric(matched[OUTLAY_COL], errors='coerce')
            n_non_null  = outlay_vals.notna().sum()
            n_nonzero   = (outlay_vals.fillna(0) != 0).sum()
            print(f'  {OUTLAY_COL}: {n_non_null} non-null, {n_nonzero} non-zero')
            if n_nonzero > 0:
                print(f'  Sample outlay values: {outlay_vals[outlay_vals != 0].head(5).tolist()}')
        else:
            print(f'  OUTLAY_COL "{OUTLAY_COL}" not in matched columns: {list(matched.columns)}')

## 3. Full Fetch

**Run this section only if the probe above confirmed:**
1. FAIN matches work (probe returned matched rows)
2. The outlay column exists and is populated for at least some records

If FAINs matched but outlay column is always null, the bulk download does not carry outlay data for assistance awards — document as a confirmed gap.

**Download strategy:** the top 5 agencies (State, USAID, NSF, NEH, IMLS) account for 76% of our grants. Download per-agency filtered files for action_date 2010–2025 to keep individual file sizes manageable. The remaining agencies are batched together in a final catch-all request.

Each download is processed immediately: stream-parse, filter to our FAINs, save matched rows to JSONL. Intermediate CSVs are deleted after parsing.

In [ ]:
# ── Confirm probe before full fetch ───────────────────────────────────────────
# After reviewing probe output, set these to the confirmed correct values.
# FAIN_COL and OUTLAY_COL were set in the probe section above — update if needed.

print('Confirmed column names:')
print(f'  FAIN column:       {FAIN_COL}')
print(f'  Outlay column:     {OUTLAY_COL}')
print(f'  Obligation column: {OBLIGATION_COL}')
print()
print('If these are wrong based on probe output, update FAIN_COL / OUTLAY_COL / OBLIGATION_COL')
print('in the probe section (cell above the schema inspection) and re-run before proceeding.')

In [ ]:
# ── Download plan: pre-built annual archive files ────────────────────────────
# Bypasses the bulk download generation API (which drops connections).
# Files are hosted at files.usaspending.gov — static CDN, reliable.
# FY = federal fiscal year (Oct 1 – Sep 30). FY2025 = Oct 2024 – Sep 2025.
#
# Strategy: check which numbered files exist per fiscal year, then download
# each one, stream-parse, filter to our FAIN set.

# Our grants span mostly 2020-2025 calendar years. Cover FY2020–FY2026 to
# capture all action dates regardless of where the fiscal year boundary falls.
FISCAL_YEARS = list(range(2020, 2027))

print('Probing available archive files...')
archive_plan = []
for fy in FISCAL_YEARS:
    files = archive_file_urls(fy)
    archive_plan.extend([{'fy': fy, **f} for f in files])
    status = f'{len(files)} file(s)' if files else 'not found'
    total_mb = sum(f['size_mb'] for f in files)
    print(f'  FY{fy}: {status}', end='')
    if files:
        print(f', total {total_mb:.0f} MB')
    else:
        print()

print(f'\nTotal files to download: {len(archive_plan)}')
total_size = sum(f['size_mb'] for f in archive_plan)
print(f'Estimated total size:    {total_size:.0f} MB (compressed)')


In [ ]:
already_matched_fains = set()
if grant_outlays_path.exists():
    for rec in load_jsonl(grant_outlays_path):
        if rec.get('fain'):
            already_matched_fains.add(rec['fain'])

remaining_fains = fain_set - already_matched_fains
print(f'FAINs total:     {len(fain_set):,}')
print(f'Already matched: {len(already_matched_fains):,}')
print(f'Remaining:       {len(remaining_fains):,}')

archive_dir = DOWNLOAD_DIR / 'archive'
archive_dir.mkdir(exist_ok=True)

for file_idx, plan in enumerate(archive_plan):
    if not remaining_fains:
        print('All FAINs matched — stopping early')
        break

    fy     = plan['fy']
    url    = plan['url']
    size   = plan['size_mb']
    fname  = url.split('/')[-1]

    print(f'\n[{file_idx+1}/{len(archive_plan)}] FY{fy} — {fname} ({size:.0f} MB compressed)')
    print(f'  {len(remaining_fains):,} FAINs remaining')

    zip_path = archive_dir / fname
    try:
        download_archive_file(url, zip_path)
    except RuntimeError as e:
        print(f'  Skipping: {e}')
        continue

    # Extract and parse
    csv_paths = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith('.csv'):
                zf.extract(name, archive_dir)
                csv_paths.append(archive_dir / name)
    zip_path.unlink()

    for csv_path in csv_paths:
        matched = filter_csv_to_fains(
            csv_path, remaining_fains, FAIN_COL, OUTLAY_COL, OBLIGATION_COL
        )
        if not matched.empty:
            keep_cols = [col for col in [FAIN_COL, OUTLAY_COL, OBLIGATION_COL] if col in matched.columns]
            trimmed = matched[keep_cols].copy()
            for col in [OUTLAY_COL, OBLIGATION_COL]:
                if col in trimmed.columns:
                    trimmed[col] = pd.to_numeric(trimmed[col], errors='coerce')

            for fain, group in trimmed.groupby(FAIN_COL):
                award_id = fain_to_id.get(fain)
                record = {
                    'award_id':         award_id,
                    'fain':             fain,
                    'total_outlays':    round(float(group[OUTLAY_COL].sum()), 2) if OUTLAY_COL in group.columns else None,
                    'total_obligation': round(float(group[OBLIGATION_COL].sum()), 2) if OBLIGATION_COL in group.columns else None,
                    'row_count':        len(group),
                    'source':           f'archive:FY{fy}',
                }
                append_jsonl(grant_outlays_path, [record])
                already_matched_fains.add(fain)
                remaining_fains.discard(fain)

            n_new = trimmed[FAIN_COL].nunique()
            print(f'  +{n_new} awards matched ({len(remaining_fains):,} FAINs remaining)')

        csv_path.unlink()

print(f'\nFetch complete.')
print(f'  Matched FAINs:   {len(already_matched_fains):,}')
print(f'  Unmatched FAINs: {len(remaining_fains):,}')
if remaining_fains:
    print(f'  Unmatched FAINs had no action_date transaction in the downloaded fiscal years.')


## 4. Assembly and Coverage

In [ ]:
if not grant_outlays_path.exists():
    raise FileNotFoundError(f'{grant_outlays_path} not found — run the full fetch first.')

outlays_df = pd.DataFrame(load_jsonl(grant_outlays_path))
print(f'Outlay records: {len(outlays_df):,}')

if 'total_outlays' in outlays_df.columns:
    n_non_null = outlays_df['total_outlays'].notna().sum()
    n_nonzero  = (outlays_df['total_outlays'].fillna(0).astype(float) > 0).sum()
    print(f'  total_outlays non-null: {n_non_null:,}')
    print(f'  total_outlays > 0:      {n_nonzero:,}')

# Merge onto grants
merged = grants_df.merge(
    outlays_df[['award_id', 'total_outlays', 'total_obligation', 'source']].rename(
        columns={'total_outlays': 'bulk_outlays', 'total_obligation': 'bulk_obligation'}
    ),
    on='award_id',
    how='left',
)

def _f(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

merged['ceiling']   = merged['value'].apply(_f)
merged['obligated'] = merged['total_obligation'].apply(_f)
merged['outlays']   = merged['bulk_outlays'].apply(_f)

n_grants    = len(merged)
n_all_three = int((
    merged['ceiling'].notna() &
    merged['obligated'].notna() &
    merged['outlays'].notna()
).sum())

print(f'\nThree-number record — grants:')
print(f'  Total:             {n_grants:,}')
print(f'  Ceiling present:   {merged["ceiling"].notna().sum():,}')
print(f'  Obligated present: {merged["obligated"].notna().sum():,}')
print(f'  Outlays present:   {merged["outlays"].notna().sum():,}')
print(f'  All three:         {n_all_three:,} ({n_all_three/(n_grants or 1)*100:.1f}%)')

# Cross-check bulk obligation vs nb02 total_obligation
check = merged[merged['obligated'].notna() & merged['bulk_obligation'].apply(_f).notna()].copy()
check['bulk_obl_f'] = check['bulk_obligation'].apply(_f)
if not check.empty:
    diff = (check['bulk_obl_f'] - check['obligated']).abs() / check['obligated'].abs().replace(0, float('nan'))
    within_1pct = (diff < 0.01).sum()
    print(f'\nObligation cross-check (bulk vs nb02): {within_1pct}/{len(check)} within 1%')

In [ ]:
def _safe_pct(a, b): return a / (b or 1) * 100
def _sum_col(df, col):
    if col not in df.columns: return None
    s = pd.to_numeric(df[col], errors='coerce')
    return round(float(s.sum()), 0) if s.notna().any() else None

agg = {
    'total_ceiling':   _sum_col(merged, 'ceiling'),
    'total_obligated': _sum_col(merged, 'obligated'),
    'total_outlays':   _sum_col(merged, 'outlays'),
}

gap_valid = merged[merged['ceiling'].notna() & merged['obligated'].notna() & (merged['ceiling'] > 0)].copy()
gap_valid['gap_pct'] = (gap_valid['ceiling'] - gap_valid['obligated']) / gap_valid['ceiling'] * 100
gap_stats = {}
if not gap_valid.empty:
    gap_stats = {
        'n':              len(gap_valid),
        'median_gap_pct': round(float(gap_valid['gap_pct'].median()), 1),
        'mean_gap_pct':   round(float(gap_valid['gap_pct'].mean()),   1),
        'pct_gt_50':      round(_safe_pct((gap_valid['gap_pct'] > 50).sum(), len(gap_valid)), 1),
    }

outlay_valid = merged[merged['outlays'].notna() & merged['obligated'].notna() & (merged['obligated'] > 0)].copy()
outlay_stats = {}
if not outlay_valid.empty:
    outlay_valid['outlay_rate'] = outlay_valid['outlays'] / outlay_valid['obligated']
    outlay_stats = {
        'n':                     len(outlay_valid),
        'median_outlay_rate':    round(float(outlay_valid['outlay_rate'].median()), 3),
        'mean_outlay_rate':      round(float(outlay_valid['outlay_rate'].mean()),   3),
        'pct_gt_80pct_disbursed': round(_safe_pct((outlay_valid['outlay_rate'] > 0.8).sum(), len(outlay_valid)), 1),
    }

print('=== Aggregate Totals — Grants ===')
for k, v in agg.items():
    print(f'  {k}: ${(v or 0):,.0f}')
print()
print(f'Three-number coverage: {n_all_three:,}/{n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%)')
if gap_stats:
    print(f"\nCeiling gap: median {gap_stats['median_gap_pct']}%, {gap_stats['pct_gt_50']}% of grants >50% unobligated at cancellation")
if outlay_stats:
    print(f"\nOutlay rate: median {outlay_stats['median_outlay_rate']:.1%}, {outlay_stats['pct_gt_80pct_disbursed']}% of grants >80% disbursed")

## 5. Publish

In [ ]:
output = {
    'source_notebooks': ['02_fetch'],
    'grants': {
        'total_assembled':        n_grants,
        'all_three_numbers':      n_all_three,
        'all_three_numbers_pct':  round(_safe_pct(n_all_three, n_grants), 1),
        'ceiling_gap':            gap_stats,
        'outlay_rate':            outlay_stats,
        'aggregate':              agg,
    },
    'outlay_source': {
        'method':   'bulk_download',
        'endpoint': 'POST /api/v2/bulk_download/awards/',
        'columns':  {'fain': FAIN_COL, 'outlays': OUTLAY_COL, 'obligation': OBLIGATION_COL},
        'strategy': 'per-agency filtered downloads, stream-parsed in chunks, joined on FAIN',
    },
    'methodology_notes': [
        'ceiling = DOGE value field',
        'obligated = USASpending total_obligation from /api/v2/awards/{id}/',
        f'outlays = {OUTLAY_COL} from Assistance_PrimeTransactions bulk download, summed per FAIN',
        'FAIN extracted from generated_unique_award_id: ASST_{TYPE}_{FAIN}_{AGENCY_CODE}',
        'grant coverage limited to ~78% of DOGE grants with a direct USASpending link',
    ],
}

output_json_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
outlay_line = (
    f'  - outlays: ${(agg["total_outlays"] or 0):,.0f} '
    f'(median outlay rate {outlay_stats.get("median_outlay_rate", 0):.1%}, '
    f'{outlay_stats.get("pct_gt_80pct_disbursed", 0)}% of grants >80% disbursed)'
) if outlay_stats else '  - outlays: not available'

findings_md = f"""# Findings — 05: Grant Outlays via Bulk Download

*Input: {n_grants:,} matched grants (from notebook 02 checkpoint).*  
*Methodology: `notebooks/05_grant_outlays_bulk.ipynb`*

## Confirmed

- **Outlay source:** `{OUTLAY_COL}` field from USASpending `Assistance_PrimeTransactions` bulk download files, aggregated per FAIN.
- **Three-number coverage — grants:** {n_all_three:,}/{n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%).
- **Aggregate totals — grants:**
  - ceiling:   ${(agg['total_ceiling'] or 0):,.0f}
  - obligated: ${(agg['total_obligated'] or 0):,.0f}
{outlay_line}
{f'- **Ceiling gap:** median {gap_stats["median_gap_pct"]}%, {gap_stats["pct_gt_50"]}% of grants had >50% unobligated at cancellation' if gap_stats else ''}

**Methodology constraints:**
- Outlay data is transaction-level (`federal_action_obligation` transactions summed per FAIN). The bulk file contains modification-level records; aggregation sums all transactions. This may differ from award-level cumulative figures if modifications overlap.
- Grant coverage is limited to ~78% of DOGE grant records with a direct USASpending link.

## Open

- Unmatched FAINs after bulk download: {len(remaining_fains):,} — these grants have no outlay data via any tested path.
- Linkage path for 22% of DOGE grants with no USASpending link — unresolved.
"""

findings_md_path.write_text(findings_md)
print(findings_md)

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/raw/05_grant_outlays.jsonl',
        'data/outputs/05_grant_outlays.json',
        'data/findings/05_grant_outlays.md',
    ],
    message='Grant outlays via bulk download',
    repo_dir=REPO,
)